In [ ]:
reset

In [ ]:
import os
import csv

import numpy as np
import pandas as pd
import xarray as xr
import scipy.io as sio

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from shapely.geometry.polygon import LinearRing

import matplotlib as mpl
import matplotlib.pyplot as plt
import cmocean.cm as cmo

# settings
%config InlineBackend.figure_format = 'retina'

# Proxy data. Override with PROXY_DATA_DIR; see config/paths.env.example. The default is the
# repo-relative tree, so this notebook runs with no setup provided it is launched from the repo
# root. Matches the convention in fig2 and the LIG/LGM notebooks.
dpath0 = os.environ.get('PROXY_DATA_DIR', 'data/raw')
# Observational data (OIPC isoscape) — large, stays outside the repo.
obspath = os.environ.get('OBS_DATA_DIR', 'data/external')
# Repo-tracked external data (LR04 benthic stack).
extern = 'data/external'
# save figs here
opath = os.environ.get('FIG_OUTPUT_DIR')
if not opath:
    raise RuntimeError(
        "FIG_OUTPUT_DIR is not set. Run `source config/paths.env` before launching Jupyter "
        "(see config/paths.env.example). Refusing to guess a default: the old fallback wrote to "
        "a repo-local outputs/ that tools/sync_manuscript_figs.sh never read, and figures "
        "silently diverged between the two directories."
    )
os.makedirs(opath, exist_ok=True)

In [ ]:
# read in proxy data

# nh22p
filen=f'{dpath0}/NH22P/nh22p_processed_dD_handpicked.xlsx'
# load raw and ivc data as xarray
nh22p = pd.read_excel(filen, sheet_name='Sheet1').set_index('age').to_xarray()
# load dDp data
nh22p_dDp=pd.read_excel(filen, sheet_name='dDp', header=None).values
# create a new coordinate for dDp ensemble number
n_dDp_ensemble=nh22p_dDp.shape[1] 
nh22p=nh22p.assign_coords(ensemble_n_dDp=np.arange(n_dDp_ensemble))
# add dDp to xarray
nh22p['dDp'] = (('age', 'ensemble_n_dDp'), nh22p_dDp)
# load %JAS data
nh22p_pJAS=pd.read_excel(filen, sheet_name='pJAS', header=None).values
# create a new coordinate for pJAS ensemble number
n_pJAS_ensemble=nh22p_pJAS.shape[1] 
nh22p=nh22p.assign_coords(ensemble_n_pJAS=np.arange(n_pJAS_ensemble))
# add pJAS to xarray
nh22p['pJAS'] = (('age', 'ensemble_n_pJAS'), nh22p_pJAS)


# dsdp-480-479
filen=f'{dpath0}/DSDP-480-479/d480_d479_processed_dD.xlsx'
d480=pd.read_excel(filen, sheet_name='Sheet1').set_index('age').to_xarray()
# load dDp data
d480_dDp=pd.read_excel(filen, sheet_name='dDp', header=None).values
# create a new coordinate for dDp ensemble number
n_dDp_ensemble=d480_dDp.shape[1] 
d480=d480.assign_coords(ensemble_n_dDp=np.arange(n_dDp_ensemble))
# add dDp to xarray
d480['dDp'] = (('age', 'ensemble_n_dDp'), d480_dDp)
# load %JAS data
d480_pJAS=pd.read_excel(filen, sheet_name='pJAS', header=None).values
# create a new coordinate for pJAS ensemble number
n_pJAS_ensemble=d480_pJAS.shape[1] 
d480=d480.assign_coords(ensemble_n_pJAS=np.arange(n_pJAS_ensemble))
# add pJAS to xarray
d480['pJAS'] = (('age', 'ensemble_n_pJAS'), d480_pJAS)


# lr04
# Read from data/external/lr04.mat, the same canonical copy the MATLAB pipeline's icevolcorr
# uses. `delob` columns are [age (ka), d18O (per mil), error]. This replaces the old
# LR04stack_d18O.csv, which lived on a OneDrive path that no longer exists — see DATA_MANIFEST.md.
delob = sio.loadmat(f'{extern}/lr04.mat')['delob']
lr04_age = delob[:, 0]
lr04_d18O = delob[:, 1]

In [ ]:
# DIFFS FOR dDp TIMESERIES
# -----> DON'T USE
"""
# create dictionary of age bounds (ka)
age_bnds = {}
age_bnds['hol']=[0,4]
age_bnds['lgm']=[18,24]
age_bnds['lig']=[117,130]
age_bnds['pgm']=[135,150]

# calculate mean values
dDtimeslice = {}

int_mean = { 'hol':{'d480':{}, 'nh22p':{}},
             'lgm':{'d480':{}, 'nh22p':{}},
             'lig':{'d480':{}, 'nh22p':{}}, 
             'pgm':{'d480':{}, 'nh22p':{}} }

int_err = { 'hol':{'d480':{}, 'nh22p':{}},
            'lgm':{'d480':{}, 'nh22p':{}},
            'lig':{'d480':{}, 'nh22p':{}},
            'pgm':{'d480':{}, 'nh22p':{}} }

for interval in np.array(['hol','lgm','lig','pgm']):
    t_min=age_bnds[interval][0]
    t_max=age_bnds[interval][1]
    dDtimeslice[interval]={}
    dDtimeslice[interval]['nh22p']=nh22p.sel(age=slice(t_min,t_max))
    dDtimeslice[interval]['d480']=d480.sel(age=slice(t_min,t_max))
    for core in np.array(['d480','nh22p']):
        int_mean[interval][core]=dDtimeslice[interval][core].dDp.mean(dim=['ensemble_n_dDp','age'])
        int_err[interval][core]=dDtimeslice[interval][core].dDp.std(dim=['ensemble_n_dDp','age'])
        print(f'{interval} {core} mean = {np.round(int_mean[interval][core],2).values}')
        print(f'{interval} {core} err = {np.round(int_err[interval][core],2).values}')
        
# print interval differences
for core in np.array(['d480','nh22p']):
    print(f'\n{core}')
    for interval in np.array(['lgm','lig','pgm']):
        diff = np.round(int_mean[interval][core].values - int_mean['hol'][core].values,2)
        print(f'{interval}-hol = {diff}')
"""

In [ ]:
# DIFFS FOR dDC30 TIMESERIES

# create dictionary of age bounds (ka)
age_bnds = {}
age_bnds['hol']=[0,4]
age_bnds['lgm']=[18,24]
age_bnds['lig']=[117,130]
age_bnds['pgm']=[135,150]

# calculate mean values
dDtimeslice = {}

int_mean = { 'hol':{'d480':{}, 'nh22p':{}},
             'lgm':{'d480':{}, 'nh22p':{}},
             'lig':{'d480':{}, 'nh22p':{}}, 
             'pgm':{'d480':{}, 'nh22p':{}} }

int_err = { 'hol':{'d480':{}, 'nh22p':{}},
            'lgm':{'d480':{}, 'nh22p':{}},
            'lig':{'d480':{}, 'nh22p':{}},
            'pgm':{'d480':{}, 'nh22p':{}} }

print('dDraw diffs\n')
for interval in np.array(['hol','lgm','lig','pgm']):
    t_min=age_bnds[interval][0]
    t_max=age_bnds[interval][1]
    dDtimeslice[interval]={}
    dDtimeslice[interval]['nh22p']=nh22p.sel(age=slice(t_min,t_max))
    dDtimeslice[interval]['d480']=d480.sel(age=slice(t_min,t_max))
    for core in np.array(['d480','nh22p']):
        int_mean[interval][core]=dDtimeslice[interval][core].dDraw.mean(dim=['age'])
        int_err[interval][core]=dDtimeslice[interval][core].dDraw.std(dim=['age'])
        print(f'{interval} {core} mean = {np.round(int_mean[interval][core],2).values}')
        print(f'{interval} {core} err = {np.round(int_err[interval][core],2).values}')
        
# print interval differences
for core in np.array(['d480','nh22p']):
    print(f'\n{core}')
    for interval in np.array(['lgm','lig','pgm']):
        diff = np.round(int_mean[interval][core].values - int_mean['hol'][core].values,2)
        print(f'{interval}-hol = {diff}')

In [ ]:
print('d480 max')
print(d480.dDraw.max().values)
print('\nd480 min')
print(d480.dDraw.min().values)
print('\nnh22p max')
print(nh22p.dDraw.max().values)
print('\nnh22p min')
print(nh22p.dDraw.min().values)

In [ ]:
float(int_mean['lig']['nh22p'].values)

In [ ]:
### create data frame for exporting
# Writes to data/processed/ (moved from proxy_data/, Aug 2025). Path is relative to the repo
# root, which is where notebooks must be launched from. These CSVs are tracked in git — they
# are how the Casper clone receives proxy numbers without raw proxy data crossing to /glade.
data = [
    ["core_name","lon","lat","holocene_dD","holocene_dD_1serr","lgm_dD","lgm_dD_1serr","lig_dD","lig_dD_1serr"],
    ["DSDP_480_479","-111.62","27.85",
         f"{float(int_mean['hol']['d480'].values)}",f"{float(int_err['hol']['d480'].values)}",
         f"{float(int_mean['lgm']['d480'].values)}",f"{float(int_err['lgm']['d480'].values)}",
         f"{float(int_mean['lig']['d480'].values)}",f"{float(int_err['lig']['d480'].values)}"],
    ["NH22P","-106.5183","22.5183",
         f"{float(int_mean['hol']['nh22p'].values)}",f"{float(int_err['hol']['nh22p'].values)}",
         f"{float(int_mean['lgm']['nh22p'].values)}",f"{float(int_err['lgm']['nh22p'].values)}",
         f"{float(int_mean['lig']['nh22p'].values)}",f"{float(int_err['lig']['nh22p'].values)}"],
]
with open('data/processed/timeslice_mean_proxy_dDraw.csv', "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerows(data)
"""
data = {
    'hol_0_4_ka': [-51.24, -59.34],
    'lgm_18_24_ka': [-58.98, -54.33],
    'lig_117_130_ka': [-51.05, -48.04],
    'core_lon': [-111.62, -106.5183],
    'core_lat': [27.85, 22.5183],
    'core_name': ['DSDP_480_479', 'NH22P']
}
 
# Create DataFrame
df = pd.DataFrame(data)

# Export DataFrame to CSV
df.to_csv('data/processed/timeslice_mean_proxy_dD.csv', index=False)  # index=False excludes row numbers from the CSV
"""

In [ ]:
# get modern dDp values from OIPC
lon_min=-125
lon_max=-85
lat_min=10
lat_max=42

filen='OIPC_monthly_data.nc'
oipc=xr.open_dataset(f'{obspath}/{filen}').isotopes[::-1,:,:].sel(Lon=slice(lon_min,lon_max), Lat=slice(lat_min,lat_max))
oipc=oipc.rename({'Lat':'lat','Lon':'lon'})
oipc=oipc.transpose('month','lat','lon')

# calculate unweighted monthly dD values (assumes OIPC data is already flux-weighted)
# init dicitionaries
oipc_seas = {}
Guaymas = {}
Mazatlan = {}

# get seasonal means of oipc data
oipc_seas['djf'] = oipc.sel(month=[1,2,12]).mean(dim='month')
oipc_seas['jfm'] = oipc.sel(month=[1,2,3]).mean(dim='month')
oipc_seas['jas'] = oipc.sel(month=[7,8,9]).mean(dim='month')
oipc_seas['jjas'] = oipc.sel(month=[6,7,8,9]).mean(dim='month')

# bounds around core sites
Guaymas_bnds = [-113,-109,26,29.5]
Mazatlan_bnds = [-108,-104,21,24.5]

for season in np.array(['djf','jfm','jas','jjas']):
    Guaymas[season] = oipc_seas[season].sel(lon=slice(Guaymas_bnds[0],Guaymas_bnds[1]),
                                            lat=slice(Guaymas_bnds[2],Guaymas_bnds[3])).mean(dim=['lat','lon'])
    Mazatlan[season] = oipc_seas[season].sel(lon=slice(Mazatlan_bnds[0],Mazatlan_bnds[1]),
                                             lat=slice(Mazatlan_bnds[2],Mazatlan_bnds[3])).mean(dim=['lat','lon'])


#modern_Mazatlan = (-29.5 + -32.4 + -33.2) / 3

## dDraw timeseries

In [ ]:
cold_mis_boundaries = np.array([
    [14,29,-134,-131],
    [38,45,-134,-131],
    [57,71,-134,-131],
    [84,95,-134,-131],
    [105,114,-134,-131],
    [135,141,-134,-131],
    [141,150,-134,-131]
])
cold_mis_labels = ['2', '3b', '4', '5b', '5d', '6a', '6b']
coldlabel_xpos = np.mean(cold_mis_boundaries[:,0:2], axis=1)

warm_mis_boundaries = np.array([
    [0,14,-134,-131],
    [29,38,-134,-131],
    [45,57,-134,-131],
    [71,84,-134,-131],
    [95,105,-134,-131],
    [114,135,-134,-131]
])
warm_mis_labels = ['1', '3a', '3c', '5a', '5c', '5e']
warmlabel_xpos = np.mean(warm_mis_boundaries[:,0:2], axis=1)

# start year, end year, lower dD value, upper dD value
intglcl_boundaries = np.array([
    [0,11.7,-200,0],
    [117,130,-200,0],
])
intglcl_labels = ['HOL', 'LIG']
intglcl_xpos = np.mean(intglcl_boundaries[:,0:2], axis=1)

In [ ]:
line_kw={'ls':'-', 'lw':2} #, 'marker':'s', 'mec':'k', 'mew':0.25} 
patch_kw = {'ec':'indianred', 'lw':0.5, 'linestyle':'-', 'fc':'indianred', 'alpha':0.25}
patch_kw2 = {'ec':'k', 'lw':0.5, 'linestyle':'-', 'fc':'white', 'alpha':1, 'clip_on':False}
patch_kw3 = {'ec':'k', 'lw':0.5, 'linestyle':'-', 'fc':'silver', 'alpha':1, 'clip_on':False}
tkw = {'axis':'both', 'direction':'out', 'labelsize': 11}
title_text_kw={'size':15, 'weight':'bold', 'color':'k', 'va':'center'}
label_text_kw={'size':12, 'weight':'bold', 'color':'firebrick', 'ha':'center', 'va':'bottom'} #'backgroundcolor':'white', 
label_text_kw2={'size':9, 'weight':'bold', 'color':'k', 'ha':'center', 'va':'center'}
laxis_text_kw={'weight':'normal', 'rotation':90, 'size':11, 'color':'k'}
raxis_text_kw={'weight':'normal', 'rotation':270, 'size':11, 'color':'grey'}
legend_kw = {'loc':1, 'fontsize':8, 'labelcolor':'linecolor', 'frameon':False}
# colors
sig1 = np.array([255, 196, 0]) / 255
sig2 = np.array([255, 242, 156]) / 255
colline = np.array([235, 142, 5]) / 255
# plot specs
xmin=0
xmax=150


## ++ Make Fig ++ ##
#fig, axs = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
fig = plt.figure(figsize=(9,6))
gs = fig.add_gridspec(2, hspace=0)
axs = gs.subplots(sharex=True, sharey=False)

## DSDP-480/479
# proxy data
ax1=axs[0]
ymin=-163 ; ymax=-134
ax1.text(1.5, ymax-2.5, 'DSDP-480/479', **title_text_kw)

# dDraw
ax1.fill_between(d480.age, d480.dDraw-d480.stdev, d480.dDraw+d480.stdev,
                 color=sig1, edgecolor='none', alpha=0.75, label='error')
ax1.plot(d480.age, d480.dDraw, color=colline, linewidth=1.5, label='mean')
"""
ax1.fill_between(d480.age, d480.dDivc-d480.stdev, d480.dDivc+d480.stdev,
                 color=sig1, edgecolor='none', alpha=0.65, label='error')
ax1.plot(d480.age, d480.dDivc, color=colline, linewidth=1.5, label='mean')
"""
ax1.set(xlim=[xmin,xmax], ylim=[ymin,ymax], yticks=np.arange(-163,-130,10))
ax1.set_ylabel(u'$\delta$D$_{C30}$ [‰]', labelpad=1, **laxis_text_kw)
ax1.tick_params(color='k', labelcolor='k', top=False, bottom=False, **tkw)
ax1.minorticks_on()
ax1.spines['left'].set_color('k')
ax1.spines['right'].set_color('none')
#ax1.spines['bottom'].set_color('none')

# benthic stack
ax2=ax1.twinx() 
ax2.plot(lr04_age, lr04_d18O, c='grey', **line_kw, label='LR04') # plot lr04 d18O data
ax2.set(xlabel='AGE (ka)', xlim=[xmin,xmax], ylim=[5.05,3.05], yticks=[5.0,4.5,4.0,3.5])
ax2.set_ylabel(u'LR04 $\delta^{18}O_{benthic}$ [‰]', labelpad=17, **raxis_text_kw)
ax2.tick_params(color='grey', labelcolor='grey', top=False, bottom=False, **tkw)
ax2.spines['right'].set_color('grey')
#ax2.spines['bottom'].set_color('none')

for i in np.arange(0,len(warm_mis_boundaries),1):
    pp=plt.Rectangle((warm_mis_boundaries[i,0], warm_mis_boundaries[i,2]), # bottom right corner position
                     (warm_mis_boundaries[i,1]-warm_mis_boundaries[i,0]), # box width
                     (warm_mis_boundaries[i,3]-warm_mis_boundaries[i,2]), # box height
                     zorder=0, label='_Hidden', **patch_kw2) 
    ax1.add_patch(pp)
    ax1.text(warmlabel_xpos[i], ymax+1.35, warm_mis_labels[i], **label_text_kw2)
    
for i in np.arange(0,len(cold_mis_boundaries),1):
    pp=plt.Rectangle((cold_mis_boundaries[i,0], cold_mis_boundaries[i,2]), # bottom right corner position
                     (cold_mis_boundaries[i,1]-cold_mis_boundaries[i,0]), # box width
                     (cold_mis_boundaries[i,3]-cold_mis_boundaries[i,2]), # box height
                     zorder=0, label='_Hidden', **patch_kw3) 
    ax1.add_patch(pp)
    ax1.text(coldlabel_xpos[i], ymax+1.35, cold_mis_labels[i], **label_text_kw2)
    
for i in np.arange(0,len(intglcl_boundaries),1):
    pp=plt.Rectangle((intglcl_boundaries[i,0], intglcl_boundaries[i,2]), # bottom right corner position
                     (intglcl_boundaries[i,1]-intglcl_boundaries[i,0]), # box width
                     (intglcl_boundaries[i,3]-intglcl_boundaries[i,2]), # box height
                     zorder=0, label='_Hidden', **patch_kw)
    ax1.add_patch(pp)

# re-order axes so proxy data is on top
ax1.set_zorder(ax2.get_zorder() + 1)
ax1.patch.set_visible(False)
#ax2.patch.set_visible(False)


## NH22P
# proxy data
ax3=axs[1]
ymin=-163 ; ymax=-134
ax3.text(1.5, ymax-2.5, 'NH22P', **title_text_kw)

# dDraw
ax3.fill_between(nh22p.age, nh22p.dDraw-nh22p.stdev, nh22p.dDraw+nh22p.stdev,
                 color=sig1, edgecolor='none', alpha=0.75, label='error')
ax3.plot(nh22p.age, nh22p.dDraw, color=colline, linewidth=1.5, label='mean') # mean time-series
"""
ax3.fill_between(nh22p.age, nh22p.dDivc-nh22p.stdev, nh22p.dDivc+nh22p.stdev,
                 color=sig1, edgecolor='none', alpha=0.65, label='error')
ax3.plot(nh22p.age, nh22p.dDivc, color=colline, linewidth=1.5, label='mean') # mean time-series
"""
ax3.set(xlim=[xmin,xmax], ylim=[ymin,ymax], yticks=np.arange(-163,-130,10))
ax3.set_ylabel(u'$\delta$D$_{C30}$ [‰]', labelpad=1, **laxis_text_kw)
ax3.tick_params(color='k', labelcolor='k', top=False, **tkw)
ax3.minorticks_on()
ax3.spines['left'].set_color('k')
ax3.spines['right'].set_color('none')
#ax3.spines['top'].set_color('none')
ax3.set_xlabel('AGE [ka]', color='k', weight='normal', size=11, rotation=0, labelpad=1)

# benthic stack
ax4=ax3.twinx() 
ax4.plot(lr04_age, lr04_d18O, c='grey', **line_kw, label='LR04') # plot lr04 d18O data
ax4.set(xlabel='AGE (ka)', xlim=[xmin,xmax], ylim=[5.05,3.05], yticks=[5.0,4.5,4.0,3.5])
ax4.set_ylabel(u'LR04 $\delta^{18}O_{benthic}$ [‰]', labelpad=17, **raxis_text_kw)
ax4.tick_params(color='grey', labelcolor='grey', top=False, **tkw)
ax4.spines['right'].set_color('grey')
ax4.spines['left'].set_color('none')

for i in np.arange(0,len(intglcl_boundaries),1):
    pp=plt.Rectangle((intglcl_boundaries[i,0], intglcl_boundaries[i,2]), # bottom right corner position
                     (intglcl_boundaries[i,1]-intglcl_boundaries[i,0]), # box width
                     (intglcl_boundaries[i,3]-intglcl_boundaries[i,2]), # box height
                     zorder=0, label='_Hidden', **patch_kw)
    ax3.add_patch(pp)
    ax3.text(intglcl_xpos[i], ymin+0.5, intglcl_labels[i], **label_text_kw)
    
# re-order axes so proxy data is on top
ax3.set_zorder(ax4.get_zorder() + 1)
ax3.patch.set_visible(False)

#fig.text(0.075,-.1, '''Time-series of $\delta$D$_{wax}$-inferred $\delta$D$_{prec}$.\n
#DSDP-480/479 is based on Dervla handpicked values while NH22P is based on autopick method.\n
#Two white markers on left axis indicate mean JAS and JFM $\delta$D$_{prec}$ from OIPC, weighted by IMERG precip.''')
        
#ax[1].legend(**legend_kw)

#plt.savefig(f'{opath}/dDivc_timeseries.pdf', bbox_inches='tight')

## dDp timeseries

In [ ]:
# Percentile bands are computed with np.nanpercentile directly in the plotting cell below.
# The previous version hardcoded `iters = 1000` and indexed into a sorted ensemble, which is
# only correct while both dDp sheets happen to be exactly 1000 members wide — that assumption
# silently broke once before, when the NH22P sheet was 1020 wide and the 97.5th-percentile band
# was drawn too narrow. See DATA_MANIFEST.md section 1b.

cold_mis_boundaries = np.array([
    [14,29,-38,-35],
    [38,45,-38,-35],
    [57,71,-38,-35],
    [84,95,-38,-35],
    [105,114,-38,-35],
    [135,141,-38,-35],
    [141,150,-38,-35]
])
cold_mis_labels = ['2', '3b', '4', '5b', '5d', '6a', '6b']
coldlabel_xpos = np.mean(cold_mis_boundaries[:,0:2], axis=1)

warm_mis_boundaries = np.array([
    [0,14,-38,-35],
    [29,38,-38,-35],
    [45,57,-38,-35],
    [71,84,-38,-35],
    [95,105,-38,-35],
    [114,135,-38,-35]
])
warm_mis_labels = ['1', '3a', '3c', '5a', '5c', '5e']
warmlabel_xpos = np.mean(warm_mis_boundaries[:,0:2], axis=1)

# start year, end year, lower dD value, upper dD value
intglcl_boundaries = np.array([
    [0,11.7,-100,0],
    [117,130,-100,0],
])
intglcl_labels = ['HOL', 'LIG']
intglcl_xpos = np.mean(intglcl_boundaries[:,0:2], axis=1)


In [ ]:
line_kw={'ls':'-', 'lw':2} #, 'marker':'s', 'mec':'k', 'mew':0.25} 
patch_kw = {'ec':'indianred', 'lw':0.5, 'linestyle':'-', 'fc':'indianred', 'alpha':0.25}
patch_kw2 = {'ec':'k', 'lw':0.5, 'linestyle':'-', 'fc':'white', 'alpha':1, 'clip_on':False}
patch_kw3 = {'ec':'k', 'lw':0.5, 'linestyle':'-', 'fc':'silver', 'alpha':1, 'clip_on':False}
scat_kw = {'s': 90,'c': 'w', 'marker': 'o', 'edgecolors':'k', 'alpha':1, 'zorder': 100, 'clip_on': False}
tkw = {'axis':'y', 'direction':'out', 'labelsize': 10}
title_text_kw={'size':15, 'weight':'bold', 'color':'k', 'va':'center'}
label_text_kw={'size':12, 'weight':'bold', 'color':'firebrick', 'ha':'center', 'va':'bottom'} #'backgroundcolor':'white', 
laxis_text_kw={'weight':'normal', 'rotation':90, 'size':11, 'color':'k'}
raxis_text_kw={'weight':'normal', 'rotation':270, 'size':11, 'color':'grey'}
legend_kw = {'loc':1, 'fontsize':8, 'labelcolor':'linecolor', 'frameon':False}
# colors
sig1 = np.array([255, 196, 0]) / 255
sig2 = np.array([255, 242, 156]) / 255
colline = np.array([235, 142, 5]) / 255
# plot specs
xmin=0
xmax=150


## ++ Make Fig ++ ##
#fig, axs = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
fig = plt.figure(figsize=(9,6))
gs = fig.add_gridspec(2, hspace=0)
axs = gs.subplots(sharex=True, sharey=False)

## DSDP-480/479
# proxy data
ax1=axs[0]
ymin=-75; ymax=-38
ax1.text(1.5, -41, 'DSDP-480/479', **title_text_kw)
# bands as percentiles over the ensemble axis -- correct at any ensemble width
p480 = np.nanpercentile(d480_dDp, [2.5, 16, 84, 97.5], axis=1) # -> (4, n_age)
ax1.fill_between(d480.age, p480[0], p480[3],
                 color=sig2, edgecolor='none', alpha=0.75, label='2$\sigma$') # 2-sigma shading
ax1.fill_between(d480.age, p480[1], p480[2],
                 color=sig1, edgecolor='none', alpha=0.75, label='1$\sigma$') # 1-sigma shading
ax1.plot(d480.age, np.nanmedian(d480_dDp, axis=1), color=colline, linewidth=1.5, label='mean') # mean time-series
#ax1.scatter(0, Guaymas['jfm'], label='modern', **scat_kw) 
#ax1.scatter(0, Guaymas['jjas'], label='modern', **scat_kw) 
ax1.set(xlim=[xmin,xmax], ylim=[ymin,ymax], yticks=np.arange(-70,-35,5))
ax1.set_ylabel(u'$\delta$D$_{prec}$ [‰]', labelpad=1, **laxis_text_kw)
ax1.tick_params(color='k', labelcolor='k', top=False, bottom=False, **tkw)
ax1.minorticks_on()
ax1.spines['left'].set_color('k')
ax1.spines['right'].set_color('none')
#ax1.spines['bottom'].set_color('none')

# benthic stack
ax2=ax1.twinx() 
ax2.plot(lr04_age, lr04_d18O, c='grey', **line_kw, label='LR04') # plot lr04 d18O data
ax2.set(xlabel='AGE (ka)', xlim=[xmin,xmax], ylim=[5.05,3.05], yticks=[5.0,4.5,4.0,3.5])
ax2.set_ylabel(u'LR04 $\delta^{18}O_{benthic}$ [‰]', labelpad=17, **raxis_text_kw)
ax2.tick_params(color='grey', labelcolor='grey', top=False, bottom=False, **tkw)
ax2.spines['right'].set_color('grey')
#ax2.spines['bottom'].set_color('none')

for i in np.arange(0,len(warm_mis_boundaries),1):
    pp=plt.Rectangle((warm_mis_boundaries[i,0], warm_mis_boundaries[i,2]), # bottom right corner position
                     (warm_mis_boundaries[i,1]-warm_mis_boundaries[i,0]), # box width
                     (warm_mis_boundaries[i,3]-warm_mis_boundaries[i,2]), # box height
                     zorder=0, label='_Hidden', **patch_kw2) 
    ax1.add_patch(pp)
    ax1.text(warmlabel_xpos[i], ymax+1.35, warm_mis_labels[i], **label_text_kw2)
    
for i in np.arange(0,len(cold_mis_boundaries),1):
    pp=plt.Rectangle((cold_mis_boundaries[i,0], cold_mis_boundaries[i,2]), # bottom right corner position
                     (cold_mis_boundaries[i,1]-cold_mis_boundaries[i,0]), # box width
                     (cold_mis_boundaries[i,3]-cold_mis_boundaries[i,2]), # box height
                     zorder=0, label='_Hidden', **patch_kw3) 
    ax1.add_patch(pp)
    ax1.text(coldlabel_xpos[i], ymax+1.35, cold_mis_labels[i], **label_text_kw2)
    
for i in np.arange(0,len(intglcl_boundaries),1):
    pp=plt.Rectangle((intglcl_boundaries[i,0], intglcl_boundaries[i,2]), # bottom right corner position
                     (intglcl_boundaries[i,1]-intglcl_boundaries[i,0]), # box width
                     (intglcl_boundaries[i,3]-intglcl_boundaries[i,2]), # box height
                     zorder=0, label='_Hidden', **patch_kw)
    ax1.add_patch(pp)

# re-order axes so proxy data is on top
ax1.set_zorder(ax2.get_zorder() + 1)
ax1.patch.set_visible(False)
#ax2.patch.set_visible(False)


## NH22P
# proxy data
ax3=axs[1]
ymin=-73; ymax=-38
ax3.text(1.5, -41, 'NH22P', **title_text_kw)
p22p = np.nanpercentile(nh22p_dDp, [2.5, 16, 84, 97.5], axis=1) # -> (4, n_age)
ax3.fill_between(nh22p.age, p22p[0], p22p[3], 
                color=sig2, edgecolor='none', alpha=0.75, label='2$\sigma$') # 2-sigma shading
ax3.fill_between(nh22p.age, p22p[1], p22p[2], 
                color=sig1, edgecolor='none', alpha=0.75, label='1$\sigma$') # 1-sigma shading
ax3.plot(nh22p.age, np.nanmedian(nh22p_dDp, axis=1), color=colline, linewidth=1.5, label='mean') # mean time-series
#ax3.scatter(0, Mazatlan['jfm'], label='modern', **scat_kw) 
#ax3.scatter(0, Mazatlan['jjas'], label='modern', **scat_kw) 
ax3.set(xlim=[xmin,xmax], ylim=[ymin,ymax], yticks=np.arange(-70,-35,5))
ax3.set_ylabel(u'$\delta$D$_{prec}$ [‰]', labelpad=1, **laxis_text_kw)
ax3.tick_params(color='k', labelcolor='k', top=False, **tkw)
ax3.minorticks_on()
ax3.spines['left'].set_color('k')
ax3.spines['right'].set_color('none')
#ax3.spines['top'].set_color('none')
ax3.set_xlabel('AGE [ka]', color='k', weight='normal', size=11, rotation=0, labelpad=1)

# benthic stack
ax4=ax3.twinx() 
ax4.plot(lr04_age, lr04_d18O, c='grey', **line_kw, label='LR04') # plot lr04 d18O data
ax4.set(xlabel='AGE (ka)', xlim=[xmin,xmax], ylim=[5.05,3.05], yticks=[5.0,4.5,4.0,3.5])
ax4.set_ylabel(u'LR04 $\delta^{18}O_{benthic}$ [‰]', labelpad=17, **raxis_text_kw)
ax4.tick_params(color='grey', labelcolor='grey', top=False, **tkw)
ax4.spines['right'].set_color('grey')
ax4.spines['left'].set_color('none')

for i in np.arange(0,len(intglcl_boundaries),1):
    pp=plt.Rectangle((intglcl_boundaries[i,0], intglcl_boundaries[i,2]), # bottom right corner position
                     (intglcl_boundaries[i,1]-intglcl_boundaries[i,0]), # box width
                     (intglcl_boundaries[i,3]-intglcl_boundaries[i,2]), # box height
                     zorder=0, label='_Hidden', **patch_kw)
    ax3.add_patch(pp)
    ax3.text(intglcl_xpos[i], ymin+0.5, intglcl_labels[i], **label_text_kw)
    
# re-order axes so proxy data is on top
ax3.set_zorder(ax4.get_zorder() + 1)
ax3.patch.set_visible(False)

#fig.text(0.075,-.1, '''Time-series of $\delta$D$_{wax}$-inferred $\delta$D$_{prec}$.\n
#DSDP-480/479 is based on Dervla handpicked values while NH22P is based on autopick method.\n
#Two white markers on left axis indicate mean JAS and JFM $\delta$D$_{prec}$ from OIPC.''')
        
#ax[1].legend(**legend_kw)

plt.savefig(f'{opath}/dDp_timeseries.pdf', bbox_inches='tight')

## OIPC modern dD climatology maps

In [ ]:
# Proxy Data
clons=[-106.5183, -111.62] 
clats=[22.5183, 27.85]
#ddiff=[12.42523674, -0.67940596] #might need to recalculate based on Jess' repicks
# Model Data
lon = oipc.lon
lat = oipc.lat
# plot specs
lw=1
text_kw={'color':'k', 'weight':'bold', 'size':14, 'ha':'center', 'va':'bottom'}
text_kw1={'color':'k', 'weight':'bold', 'size':12, 'ha':'left', 'va':'bottom'}
titles=np.array(['$\delta$D$_{precip}$ JFM', '$\delta$D$_{precip}$ JJAS'])
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-120., -95., 15., 35.]
# isotopes cmap
icmap=cmo.solar
ivmin=-90
ivmax=-10
ilevels=np.linspace(ivmin, ivmax, 21)
inorm=mpl.colors.BoundaryNorm(ilevels, icmap.N)


# ------------------- #
#      Make Plot      #
# ------------------- #
months=['January-February-March', 'June-July-August-September']
t_months=months[1]

fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(11,5), layout='constrained', subplot_kw={'projection': proj})

# jfm
cf1=ax[0].pcolormesh(lon, lat, oipc_seas['jfm'], cmap=icmap, norm=inorm, transform=trans)
ax[0].scatter(x=clons, y=clats, c='w', alpha=1, edgecolor='k', s=150, transform=trans, zorder=100)

# jas
ax[1].pcolormesh(lon, lat, oipc_seas['jjas'], cmap=icmap, norm=inorm, transform=trans)
ax[1].scatter(x=clons, y=clats, c='w', alpha=1, edgecolor='k', s=150, transform=trans, zorder=100)


for i in [0,1]:
    ax[i].coastlines()
    ax[i].add_feature(cfeature.BORDERS)
    ax[i].add_feature(cfeature.STATES, linewidth=0.5)
    ax[i].text(map_bnds[0], map_bnds[3]+0.5, titles[i], **text_kw1)
    ring=LinearRing(list(zip([-113.,-109,-109,-113.], [29.5,29.5,26,26])))
    ax[i].add_geometries([ring], crs=trans, fc='none', ec='k', lw=1, linestyle='--', zorder=9)
    ring=LinearRing(list(zip([-108.,-104,-104,-108.], [24.5,24.5,21,21])))
    ax[i].add_geometries([ring], crs=trans, fc='none', ec='k', lw=1, linestyle='--', zorder=9)
    ax[i].set_extent(map_bnds, crs=trans)
    if i==0:
        gl=ax[i].gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.right_labels=False
    if i==1:
        gl=ax[i].gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.left_labels=False; gl.right_labels=False

cbar_ax1 = fig.add_axes([1, .1, 0.025, 0.8])
cbar1 = fig.colorbar(cf1, ticks=np.arange(-100,10,10), orientation='vertical', extend='both', cax=cbar_ax1)
cbar1.set_label('[$\delta$D$_{p}$]', weight='normal', labelpad=15, rotation=270)
cbar1.ax.tick_params(labelsize=10)
for tick in cbar1.ax.xaxis.get_major_ticks():
    tick.label1.set_fontweight('normal')

fig.text(0,0, r'Spatial variability of continental dDp from OIPC.')

#plt.savefig("cesm1.2_LIG-PI_jas_dDp_precip.pdf", bbox_inches='tight')